In [1]:
from pymongo import MongoClient
import pandas as pd
from datetime import datetime

In [2]:
import sqlite3
import requests
from bs4 import BeautifulSoup
from datetime import datetime

In [5]:

import sqlite3

def criar_banco():
    conexao = sqlite3.connect('noticias.db')
    cursor = conexao.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS noticias (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo TEXT NOT NULL,
            link TEXT,
            data_coleta TEXT NOT NULL
        )
    ''')
    conexao.commit()
    conexao.close()

criar_banco()  

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from datetime import datetime

def coletar_manchetes(url):
    resposta = requests.get(url) 
    soup = BeautifulSoup(resposta.text, "html.parser")  
    
    noticias = []
    for elemento in soup.find_all(["h1", "h2", "h3", "a"]):  
        titulo = elemento.text.strip()
        if len(titulo) < 25:  # Ignora títulos muito curtos
            continue
        
        # Tenta pegar o link associado ao título
        link = urljoin(url, elemento.get("href", url))
        noticia = {"titulo": titulo, "link": link, "data_coleta": str(datetime.now())}
        noticias.append(noticia)

        if len(noticias) >= 10:  #
            break

    return noticias

In [7]:
def salvar_no_banco(noticias):
    conexao = sqlite3.connect('noticias.db')
    cursor = conexao.cursor()

    for noticia in noticias:
        cursor.execute('''
            INSERT INTO noticias (titulo, link, data_coleta)
            VALUES (?, ?, ?)
        ''', (noticia['titulo'], noticia['link'], noticia['data_coleta']))

    conexao.commit()
    conexao.close()

In [9]:
def main():
    criar_banco()  # Cria o banco de dados

    url = "https://www.playstation.com/pt-br/"  # Link do site
    noticias = coletar_manchetes(url)  # Coleta as manchetes
    salvar_no_banco(noticias)  # Salva no banco de dados

    print("Manchetes coletadas e salvas com sucesso!")

if __name__ == "__main__":
    main()

Manchetes coletadas e salvas com sucesso!
